# Compare the saved financial RAG results

Recompute the main report metrics from the 156 saved outcomes. The notebook reads CSV files locally and makes no model or network calls.

In [1]:
from pathlib import Path
import csv
import statistics
from IPython.display import Markdown, display

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "results/evaluation_by_question.csv").exists())
with (root / "results/evaluation_by_question.csv").open(newline="") as file:
    outcomes = list(csv.DictReader(file))
policies = {v: [r for r in outcomes if r["variant"] == v] for v in ("B0", "B1", "B2")}
questions = {r["financebench_id"] for r in outcomes}
assert len(outcomes) == 156 and len(questions) == 52
assert all(len(rows) == 52 and {r["financebench_id"] for r in rows} == questions
           for rows in policies.values())

## Retrieval, supported answers and cost

A supported pass requires correct content, citation support and citation completeness. Page recall counts the annotated evidence pages represented by the final passages.

In [2]:
criteria = ("answer_correctness", "citation_support", "citation_completeness")
summary = [
    "| Policy | Mean page recall | Supported passes | Search rounds | Model tokens |",
    "| --- | ---: | ---: | ---: | ---: |",
]
observed = {}
for variant, rows in policies.items():
    recall = statistics.mean(float(r["evidence_page_recall"]) for r in rows)
    supported = sum(all(r[key] == "pass" for key in criteria) for r in rows)
    rounds = sum(int(r["retrieval_rounds"]) for r in rows)
    tokens = sum(int(r["input_tokens"]) + int(r["output_tokens"]) for r in rows)
    observed[variant] = (supported, rounds, tokens)
    summary.append(f"| {variant} | {recall:.1%} | {supported}/{len(rows)} | {rounds} | {tokens:,} |")

assert observed == {"B0": (6, 52, 165773), "B1": (10, 104, 292545), "B2": (14, 92, 308831)}
display(Markdown("\n".join(summary)))

| Policy | Mean page recall | Supported passes | Search rounds | Model tokens |
| --- | ---: | ---: | ---: | ---: |
| B0 | 25.0% | 6/52 | 52 | 165,773 |
| B1 | 34.6% | 10/52 | 104 | 292,545 |
| B2 | 48.1% | 14/52 | 92 | 308,831 |

## Interpretation

B2 retrieved more annotated pages and used more model tokens. The four-pass gain over B1 has a paired interval that includes zero. The scores come from saved AI ratings without measured human agreement. See [results and findings](../results/FINDINGS.md) for the separate development comparisons and their earlier scoring method.